# ShimiStudio — Cloud Worker v4.2 (חינם)
**VM worker על GPU של Google Colab — מתחבר לתור ה-render של ShimiStudio ומרנדר לבד**

הוראות:
1. לחץ על `Runtime` → `Run all`
2. חכה ~5 דקות (התקנה + הורדת מודלים)
3. הworker יירשם לתור ויתפוס את הjob הממתין אוטומטית
4. הוידאו יועלה חזרה לסטודיו בסיום

⚠️ לפני Run all: Runtime → Change runtime type → T4 GPU

In [ ]:
# 1️⃣ בדיקת GPU
import torch
assert torch.cuda.is_available(), '❌ אין GPU! Runtime → Change runtime type → T4 GPU'
print('✅ GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')

In [ ]:
# 2️⃣ התקנת ComfyUI
import os
COMFY = '/content/ComfyUI'
if not os.path.exists(COMFY):
    !git clone -q --depth 1 https://github.com/comfyanonymous/ComfyUI {COMFY}
os.chdir(COMFY)
!pip install -q -r requirements.txt
print('✅ ComfyUI מותקן')

In [ ]:
# 3️⃣ Custom Nodes — AnimateDiff Evolved (לצומת AnimateDiffLoaderV1)
import os
CUSTOM = '/content/ComfyUI/custom_nodes'
!git clone -q --depth 1 https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved {CUSTOM}/ComfyUI-AnimateDiff-Evolved
print('✅ AnimateDiff Evolved מותקן')

In [ ]:
# 4️⃣ הורדת מודלים (~6GB, כ-3 דקות)
import urllib.request, os

MODELS = {
    # Checkpoint SD1.5 — DreamShaper (ציבורי, בלי מפתח)
    '/content/ComfyUI/models/checkpoints/DreamShaper62.safetensors':
        'https://huggingface.co/Yntec/DreamShaper/resolve/main/DreamShaper62.safetensors',
    # AnimateDiff motion model
    '/content/ComfyUI/models/animate_diff_models/mm_sd_v15_v2.ckpt':
        'https://huggingface.co/guoyww/animatediff/resolve/main/mm_sd_v15_v2.ckpt',
}
for path, url in MODELS.items():
    if os.path.exists(path):
        print('קיים:', os.path.basename(path)); continue
    print('מוריד:', os.path.basename(url))
    urllib.request.urlretrieve(url, path)
    print('  ✅', round(os.path.getsize(path)/1e9,1), 'GB')
print('✅ מודלים מוכנים')

In [ ]:
# 5️⃣ Worker v4.2 מהריפו + הגדרות
import urllib.request, json, os
os.chdir('/content')
urllib.request.urlretrieve('https://raw.githubusercontent.com/sunraz/ShimiStudio/main/worker.py', '/content/worker.py')

CONFIG = {
    "token": "colab-free-vm-01",
    "name": "Colab-T4-Free",
    "apiBase": "https://preview-sandbox--6a9206e9f29b8d9f70a77b47.base44.app/api/apps/6a9206e9f29b8d9f70a77b47"
}
with open('/content/config.json','w') as f:
    json.dump(CONFIG, f, indent=2)
print('✅ worker v4.2 + config מוכנים')

In [ ]:
# 6️⃣ הפעלת ComfyUI ברקע
import subprocess, time, urllib.request, os

os.chdir('/content/ComfyUI')
subprocess.Popen(
    ['python','main.py','--listen','127.0.0.1','--port','8188','--disable-auto-launch'],
    stdout=open('/content/comfy.log','w'), stderr=subprocess.STDOUT)

for i in range(60):
    time.sleep(5)
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=3)
        print('✅ ComfyUI ONLINE'); break
    except:
        if i%6==0: print('ממתין...', (i+1)*5, 'שניות')
else:
    print(open('/content/comfy.log').read()[-2000:]); raise SystemExit('ComfyUI לא עלה')

In [ ]:
# 7️⃣ הפעלת הWorker — תופס את הjob מהתור ומרנדר
# התא הזה רץ עד שהוידאו מוכן. אפשר לעצור אחרי עם Stop.
import os
os.chdir('/content')
!python worker.py

---
## ℹ️ מידע
- הworker מתרשם לתור עם token `colab-free-vm-01`
- תופס job ממתין, מרנדר AnimateDiff (16 פריימים @ 8fps), מעלה את הוידאו לסטודיו
- session של Colab חינמי: עד 12 שעות. להרצה חוזרת — Run all שוב
- להרבה jobs רצופים: פשוט תשאיר את תא 7 רץ — הוא ממשיך לתפוס jobs מהתור